In [1]:
from typing import Callable, Literal

import torch
import torch.utils.benchmark as benchmark

assert torch.cuda.is_available()

In [2]:
def run_pass(
    func: Callable[..., torch.Tensor],
    args: tuple | list,
    do_backward: bool,
):
    out = func(*args)
    if do_backward:
        out.sum().backward()

def bench(
    *func_args_device_triples: tuple[Callable[..., torch.Tensor], tuple, Literal["cpu", "cuda"]],
    do_backward: bool = False,
):
    for func, args, device in func_args_device_triples:
        # Clear existing gradients
        for arg in args:
            if isinstance(arg, torch.Tensor) and arg.grad is not None:
                arg.grad = None

        # Move all tensors to the correct device
        args = [
            arg.to(device) if isinstance(arg, torch.Tensor) else arg
            for arg in args
        ]

        t = benchmark.Timer(
            stmt="run_pass(func, args, do_backward)",
            globals={"run_pass": run_pass, "func": func, "args": args, "do_backward": do_backward},
            label=f"Function: {func.__name__} | Device: {device.upper()} | Pass: {'Forward & Backward' if do_backward else 'Forward Only'}",
        )

        # timeit() automatically handles warmup and CUDA synchronization
        print(t.timeit(100))
        print()

### Matrix Multiplication over GF(2)

In [3]:
def matmul_GF2_direct(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return (x @ y) % 2

def matmul_GF2_indirect(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return (x.float() @ y.float()).round().int() % 2


In [4]:
num_iters = 10
batch_size = 1024
num_vars = 186
num_chks = 72

rng = torch.Generator()
rng.manual_seed(42)

ehat = torch.randint(0, 2, (num_iters, batch_size, num_vars), dtype=torch.int32, generator=rng)
chkmat = torch.randint(0, 2, (num_chks, num_vars), dtype=torch.int32, generator=rng)

In [5]:
# Check correctness
torch.equal(matmul_GF2_direct(ehat, chkmat.T), matmul_GF2_indirect(ehat, chkmat.T))

True

In [6]:
bench(
    (matmul_GF2_direct, (ehat, chkmat.T), "cpu"),
    (matmul_GF2_indirect, (ehat, chkmat.T), "cpu"),
    (matmul_GF2_indirect, (ehat, chkmat.T), "cuda"),
)

Function: matmul_GF2_direct | Device: CPU | Pass: Forward Only
  14.52 ms
  1 measurement, 100 runs , 1 thread

Function: matmul_GF2_indirect | Device: CPU | Pass: Forward Only
  7.67 ms
  1 measurement, 100 runs , 1 thread

Function: matmul_GF2_indirect | Device: CUDA | Pass: Forward Only
  70.76 us
  1 measurement, 100 runs , 1 thread



### Convert LLRs to ehat

In [7]:
def f1(x):
    return torch.where(x < 0, 1.0, 0.0)

def f2(x):
    return (x < 0).float()

In [8]:
num_iters = 10
batch_size = 1024
num_vars = 186
num_chks = 72

rng = torch.Generator()
rng.manual_seed(42)

llrs = torch.randn(num_iters, batch_size, num_vars)

In [9]:
# Check correctness
torch.equal(f1(llrs), f2(llrs))

True

In [10]:
bench(
    (f1, (llrs,), "cpu"),
    (f2, (llrs,), "cpu"),
    (f1, (llrs,), "cuda"),
    (f2, (llrs,), "cuda"),
)

Function: f1 | Device: CPU | Pass: Forward Only
  8.70 ms
  1 measurement, 100 runs , 1 thread

Function: f2 | Device: CPU | Pass: Forward Only
  2.20 ms
  1 measurement, 100 runs , 1 thread

Function: f1 | Device: CUDA | Pass: Forward Only
  31.82 us
  1 measurement, 100 runs , 1 thread

Function: f2 | Device: CUDA | Pass: Forward Only
  17.63 us
  1 measurement, 100 runs , 1 thread



### Sign

In [11]:
def f1(x):
    return torch.where(x < 0, -1.0, 1.0)

def f2(x):
    return 1 - (x < 0).float() * 2

In [12]:
batch_size = 1024
num_chks = 72
max_cn_deg = 6
msg_features = 16

rng = torch.Generator()
rng.manual_seed(42)

msgs = torch.randn(batch_size, num_chks, max_cn_deg, msg_features)

In [13]:
# Check correctness
torch.equal(f1(msgs), f2(msgs))

True

In [14]:
bench(
    (f1, (msgs,), "cpu"),
    (f2, (msgs,), "cpu"),
    (f1, (msgs,), "cuda"),
    (f2, (msgs,), "cuda"),
)

Function: f1 | Device: CPU | Pass: Forward Only
  32.20 ms
  1 measurement, 100 runs , 1 thread

Function: f2 | Device: CPU | Pass: Forward Only
  15.66 ms
  1 measurement, 100 runs , 1 thread

Function: f1 | Device: CUDA | Pass: Forward Only
  220.17 us
  1 measurement, 100 runs , 1 thread

Function: f2 | Device: CUDA | Pass: Forward Only
  490.65 us
  1 measurement, 100 runs , 1 thread



### Leave-one-out Sign Product

In [15]:
def leave_one_out_sign_product_via_expansion(x: torch.Tensor, dim: int, expand_shape: tuple[int, ...], diag_mask: torch.Tensor) -> torch.Tensor:
    x_sgn = torch.where(x < 0, -1.0, 1.0)  # (..., 1, n, ...)
    return (
        x_sgn.unsqueeze(dim)  # (..., 1, n, ...)
        .expand(expand_shape)  # (..., n, n, ...)
        .masked_fill(diag_mask, 1.0)  # (..., n, n, ...)
        .prod(dim+1)  # (..., n, ...)
    )

def leave_one_out_sign_product_via_selfmul(x: torch.Tensor, dim: int) -> torch.Tensor:
    x_sgn = torch.where(x < 0, -1.0, 1.0)
    return x_sgn.prod(dim, keepdim=True) * x_sgn


In [16]:
batch_size = 256
num_chks = 72
max_cn_deg = 6
msg_features = 16

rng = torch.Generator()
rng.manual_seed(42)

msgs = torch.randn(batch_size, num_chks, max_cn_deg, msg_features)
expand_shape = (batch_size, num_chks, max_cn_deg, max_cn_deg, msg_features)  # (B, C, Δc, Δc, M)
diag_mask = torch.eye(max_cn_deg, dtype=torch.bool).unsqueeze(0).unsqueeze(0).unsqueeze(-1)  # (1, 1, Δc, Δc, 1)

In [17]:
# Check correctness
torch.equal(
    leave_one_out_sign_product_via_expansion(msgs, 2, expand_shape, diag_mask),
    leave_one_out_sign_product_via_selfmul(msgs, 2)
)

True

In [18]:
bench(
    (leave_one_out_sign_product_via_expansion, (msgs, 2, expand_shape, diag_mask), "cpu"),
    (leave_one_out_sign_product_via_selfmul, (msgs, 2), "cpu"),
    (leave_one_out_sign_product_via_expansion, (msgs, 2, expand_shape, diag_mask), "cuda"),
    (leave_one_out_sign_product_via_selfmul, (msgs, 2), "cuda"),
)

Function: leave_one_out_sign_product_via_expansion | Device: CPU | Pass: Forward Only
  45.53 ms
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_sign_product_via_selfmul | Device: CPU | Pass: Forward Only
  9.65 ms
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_sign_product_via_expansion | Device: CUDA | Pass: Forward Only
  443.14 us
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_sign_product_via_selfmul | Device: CUDA | Pass: Forward Only
  51.98 us
  1 measurement, 100 runs , 1 thread



### Leave-one-out Min

In [19]:
def leave_one_out_min_via_expansion(x: torch.Tensor, dim: int, expand_shape: tuple[int, ...], diag_mask: torch.Tensor) -> torch.Tensor:
    return (
        x.unsqueeze(dim)  # (..., 1, n, ...)
        .expand(expand_shape)  # (..., n, n, ...)
        .masked_fill(diag_mask, 1e8)  # (..., n, n, ...)
        .amin(dim+1)  # (..., n, ...)
    )

def leave_one_out_min_via_topk(x: torch.Tensor, dim: int) -> torch.Tensor:
    values, indices = x.topk(2, dim=dim, largest=False)  # (..., 2, ...), (..., 2, ...)
    min1, min2 = values.split(1, dim=dim)  # (..., 1, ...), (..., 1, ...)
    ind1 = indices.narrow(dim, 0, 1)  # (..., 1, ...)
    return min1.expand_as(x).clone().scatter_(dim, ind1, min2)


In [20]:
batch_size = 256
num_chks = 72
max_cn_deg = 6
msg_features = 16

rng = torch.Generator()
rng.manual_seed(42)

msgs = torch.randn(batch_size, num_chks, max_cn_deg, msg_features)
expand_shape = (batch_size, num_chks, max_cn_deg, max_cn_deg, msg_features)  # (B, C, Δc, Δc, M)
diag_mask = torch.eye(max_cn_deg, dtype=torch.bool).unsqueeze(0).unsqueeze(0).unsqueeze(-1)  # (1, 1, Δc, Δc, 1)

In [21]:
# Check correctness
torch.equal(
    leave_one_out_min_via_expansion(msgs, 2, expand_shape, diag_mask),
    leave_one_out_min_via_topk(msgs, 2)
)

True

In [22]:
bench(
    (leave_one_out_min_via_expansion, (msgs, 2, expand_shape, diag_mask), "cpu"),
    (leave_one_out_min_via_topk, (msgs, 2), "cpu"),
    (leave_one_out_min_via_expansion, (msgs, 2, expand_shape, diag_mask), "cuda"),
    (leave_one_out_min_via_topk, (msgs, 2), "cuda"),
)

Function: leave_one_out_min_via_expansion | Device: CPU | Pass: Forward Only
  43.29 ms
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_min_via_topk | Device: CPU | Pass: Forward Only
  21.11 ms
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_min_via_expansion | Device: CUDA | Pass: Forward Only
  379.01 us
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_min_via_topk | Device: CUDA | Pass: Forward Only
  1.45 ms
  1 measurement, 100 runs , 1 thread



In [23]:
msgs1 = msgs.clone().detach().requires_grad_(True)
msgs2 = msgs.clone().detach().requires_grad_(True)

loss1 = leave_one_out_min_via_expansion(msgs1, 2, expand_shape, diag_mask).sum()
loss1.backward()
loss2 = leave_one_out_min_via_topk(msgs2, 2).sum()
loss2.backward()

torch.equal(msgs1.grad, msgs2.grad)

True

In [24]:
msgs3 = msgs.clone().detach().requires_grad_(True)

bench(
    (leave_one_out_min_via_expansion, (msgs3, 2, expand_shape, diag_mask), "cpu"),
    (leave_one_out_min_via_topk, (msgs3, 2), "cpu"),
    (leave_one_out_min_via_expansion, (msgs3, 2, expand_shape, diag_mask), "cuda"),
    (leave_one_out_min_via_topk, (msgs3, 2), "cuda"),
    do_backward=True,
)

Function: leave_one_out_min_via_expansion | Device: CPU | Pass: Forward & Backward
  186.37 ms
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_min_via_topk | Device: CPU | Pass: Forward & Backward
  28.23 ms
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_min_via_expansion | Device: CUDA | Pass: Forward & Backward
  3.61 ms
  1 measurement, 100 runs , 1 thread

Function: leave_one_out_min_via_topk | Device: CUDA | Pass: Forward & Backward
  2.41 ms
  1 measurement, 100 runs , 1 thread

